# <center/> Photometric Classifcation of Brown Drawfs </center>
#### <center/> ASTR 154 Final Project </center>
<center/> Marylin Loritsch and Lauren Agarwala </center>


In [1]:
# import statements
from astroquery.vizier import Vizier  # import Vizier to query AllWISE catalog
from astropy.coordinates import SkyCoord  # import SkyCoord to query tables by RA & DEC
import astropy.units as u  # for specifying units during coordinate query 
from astropy.table import Table, vstack  # vstack for combining l & t dwarf tables
import numpy as np

General Description/Overview

# <center/> Data Sets </center>

Gen description


In [2]:
# The first step of our project is to use astroquery to query the Vizier data release AllWISE Source Catalog for our sources.
# We will start by obtaining the data for our sample of brown dwarfs, given in N. Skrzypek et al. (2016), 
# see: https://doi.org/10.1051/0004-6361/201527359 for the original article.
# Unfortunately this article only reports some of the WISE photometry, so we will query the AllWISE Source Catalog by coordinate to collect 
# information for these sources. Hence, we must collect the brown dwarf positions (RA & DEC) first from N. Skrzypek et al. (2016):

# Start by defining Vizier object:
v1 = Vizier(columns = ['**'])  # the '**' tells Vizier to list all columns in the catalog UPDATE PROB DONT NEED THIS
v1.ROW_LIMIT=-1  # this tells not to limit the rows returned 

# Next use Vizier.get_catalogs to query Vizier for the catlog we're interested in
bd_catalog = v1.get_catalogs('J/A+A/589/A49')  # J/A+A/589/A49 is the name of the catalog given by N. Skrzypek et al. (2016)

# This catalog contains 3 tables, one with information for the L-dwarfs identitfied in WISE, one for the T-dwarfs, and one for references
# We are interested in the tables containing information for the L & T dwarfs:
ldwarf_table = bd_catalog[0]  # L-dwarf table is 0th table in catalog
tdwarf_table = bd_catalog[1]  # T-dwarf table is 1st table in catalog

ldwarf_pos_table = ldwarf_table['Jmag', 'e_Jmag', 'Hmag', 'e_Hmag', 'Kmag', 'e_Kmag',
                                'W1mag', 'e_W1mag', 'W2mag', 'e_W2mag']
tdwarf_pos_table = tdwarf_table['Jmag', 'e_Jmag', 'Hmag', 'e_Hmag', 'Kmag', 'e_Kmag',
                                'W1mag', 'e_W1mag', 'W2mag', 'e_W2mag']

bd_phot_table = vstack([ldwarf_pos_table, tdwarf_pos_table], metadata_conflicts = 'silent')
bd_phot_table['Label'] = 'BD'
bd_phot_table

# NEED TO FINISH COMMENTING

Jmag,e_Jmag,Hmag,e_Hmag,Kmag,e_Kmag,W1mag,e_W1mag,W2mag,e_W2mag,Label
mag,mag,mag,mag,mag,mag,mag,mag,mag,mag,
float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,str2
17.27,0.03,16.51,0.03,15.92,0.03,--,--,--,--,BD
14.76,0.01,14.08,0.01,13.54,0.01,13.33,0.02,13.02,0.03,BD
15.46,0.01,14.48,0.01,13.62,0.01,12.97,0.02,12.54,0.02,BD
17.18,0.02,16.43,0.03,15.90,0.03,15.65,0.05,15.38,0.10,BD
16.59,0.01,15.94,0.01,15.36,0.01,15.06,0.04,14.80,0.07,BD
16.60,0.01,15.91,0.02,15.35,0.02,15.08,0.04,14.84,0.07,BD
16.95,0.02,16.27,0.02,15.75,0.02,15.40,0.04,15.20,0.09,BD
16.69,0.02,15.88,0.02,15.21,0.02,14.86,0.03,14.68,0.06,BD


In [7]:
v2 = Vizier(columns = ['Jmag', 'e_Jmag', 'Hmag', 'e_Hmag', 'Kmag', 'e_Kmag', 'W1mag', 'e_W1mag', 'W2mag', 'e_W2mag'])
v2.ROW_LIMIT= 5000
result = v2.query_constraints(catalog = 'II/328/allwise', 
                              ex = '0',  # indicates a point source
                              ccf = '0000')  # avoids contamination by various image artifacts
                              #Jmag = '!null')  # COULD CHANGE TO OR STATEMENT TO HAVE JUST 1 OF 3 BANDS to ensure at least has some 2MASS photometry (didn't require all filters because some BD are missing data, don't want to bias too much...)
nonbd_phot_table = result[0]
mask = (np.isfinite(nonbd_phot_table['Jmag']) | np.isfinite(nonbd_phot_table['Hmag']) | np.isfinite(nonbd_phot_table['Kmag']))
nonbd_phot_table = nonbd_phot_table[mask]

length = len(bd_phot_table)
random_i = np.random.choice(length, size = length, replace = False)
nonbd_phot_table = nonbd_phot_table[random_i]
nonbd_phot_table['Label'] = 'NBD'  # label for not brown dwarf

# NEED TO COMMENT

In [8]:
nonbd_phot_table

Jmag,e_Jmag,Hmag,e_Hmag,Kmag,e_Kmag,W1mag,e_W1mag,W2mag,e_W2mag,Label
mag,mag,mag,mag,mag,mag,mag,mag,mag,mag,
float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,str3
16.284,0.107,15.439,0.125,15.226,0.159,15.072,0.039,15.120,0.090,NBD
14.327,0.028,13.507,0.028,13.168,0.032,12.975,0.025,12.932,0.030,NBD
14.770,0.038,14.051,0.048,13.899,0.056,13.631,0.027,13.598,0.034,NBD
15.616,0.057,14.545,0.056,14.457,0.069,14.096,0.028,14.114,0.047,NBD
16.986,0.172,16.032,0.155,15.800,0.248,15.745,0.053,15.980,0.181,NBD
14.538,0.033,13.671,0.034,13.342,0.039,13.183,0.024,13.157,0.031,NBD
14.023,0.027,13.192,0.030,12.931,0.029,12.689,0.024,12.609,0.024,NBD
16.441,0.120,15.622,0.155,15.122,0.148,15.391,0.045,15.587,0.134,NBD


notes about the sets

 # <center/> Classification </center>

 general description on what it is, what it does, plan

In [2]:
#code

Notes about it

# <center/> MCMC </center>

general description and vibes; how it connects to classification


In [3]:
# code

notes


Ending thoughts